<a href="https://colab.research.google.com/github/KarenInfanta/openfda-adverse-event-analysis/blob/main/notebooks/02_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
DATA = Path('/content/drive/MyDrive/openfda-project/data')

# dtype=str keeps coded fields like '801' as text (no float conversion)
reports = pd.read_csv(DATA / 'processed/reports_extracted.csv', dtype=str)
drugs = pd.read_csv(DATA / 'processed/drugs_extracted.csv', dtype=str)
reactions = pd.read_csv(DATA / 'processed/reactions_extracted.csv', dtype=str)

print(reports.shape, drugs.shape, reactions.shape)

Mounted at /content/drive
(3100, 20) (11856, 8) (9335, 4)


In [4]:
# ---------- 1. Dates ----------
for col in ['receivedate', 'receiptdate']:
    reports[col + '_dt'] = pd.to_datetime(reports[col], format='%Y%m%d', errors='coerce')
print('Unparseable receivedate:', reports['receivedate_dt'].isna().sum())
print('Date range:', reports['receivedate_dt'].min().date(), 'to', reports['receivedate_dt'].max().date())
reports['report_month'] = reports['receivedate_dt'].dt.to_period('M').astype(str)

# ---------- 2. Age -> years ----------
UNIT_TO_YEARS = {'800': 10.0, '801': 1.0, '802': 1/12,
                 '803': 1/52.18, '804': 1/365.25, '805': 1/8766}
age_num = pd.to_numeric(reports['patient_age_raw'], errors='coerce')
factor = reports['patient_age_unit'].map(UNIT_TO_YEARS)
reports['age_years'] = age_num * factor

print('\nAge present but unit missing/unknown:', (age_num.notna() & factor.isna()).sum())
print('Unit present but age missing:', (age_num.isna() & reports['patient_age_unit'].notna()).sum())
invalid = reports['age_years'].notna() & ~reports['age_years'].between(0, 120)
print('Ages outside 0-120 years (set to missing):', invalid.sum())
reports.loc[invalid, 'age_years'] = np.nan
print(reports['age_years'].describe().round(1))

# Age groups (our own analysis bins, not an FDA standard)
bins = [0, 18, 45, 65, 75, 121]
labels = ['0-17', '18-44', '45-64', '65-74', '75+']
reports['age_group'] = (pd.cut(reports['age_years'], bins=bins, labels=labels, right=False)
                        .astype(object).fillna('Not recorded'))

# ---------- 3. Sex ----------
SEX_MAP = {'1': 'Male', '2': 'Female', '0': 'Unknown'}
print('\nUnexpected sex codes:', set(reports['patient_sex'].dropna()) - set(SEX_MAP))
reports['sex'] = reports['patient_sex'].map(SEX_MAP).fillna('Not recorded')

# ---------- 4. Report-level outcome (one value per report) ----------
flags = {
    'seriousnessdeath': 'Death',
    'seriousnesslifethreatening': 'Life-threatening',
    'seriousnesshospitalization': 'Hospitalization',
    'seriousnessdisabling': 'Disability',
    'seriousnesscongenitalanomali': 'Congenital anomaly',
    'seriousnessother': 'Other serious',
}
conds = [reports[c] == '1' for c in flags]
choices = list(flags.values())
conds += [reports['serious'] == '1', reports['serious'] == '2']
choices += ['Serious (type not specified)', 'Not serious']
reports['report_outcome'] = np.select(conds, choices, default='Not recorded')

any_flag = pd.concat([reports[c] == '1' for c in flags], axis=1).any(axis=1)
print("serious='2' but a seriousness flag = '1':", ((reports['serious'] == '2') & any_flag).sum())

# ---------- 5. Types, summary, save ----------
reports['n_drugs'] = reports['n_drugs'].astype(int)
reports['n_reactions'] = reports['n_reactions'].astype(int)

for c in ['sex', 'age_group', 'report_outcome']:
    print(f'\n{c}:\n', reports[c].value_counts(dropna=False))

reports.to_csv(DATA / 'processed/reports_clean.csv', index=False)
print('\nSaved reports_clean.csv with shape', reports.shape)

Unparseable receivedate: 0
Date range: 2024-01-01 to 2024-01-31

Age present but unit missing/unknown: 0
Unit present but age missing: 1
Ages outside 0-120 years (set to missing): 0
count    1824.0
mean       56.9
std        20.9
min         0.0
25%        45.0
50%        61.0
75%        73.0
max       100.0
Name: age_years, dtype: float64

Unexpected sex codes: set()
serious='2' but a seriousness flag = '1': 0

sex:
 sex
Female          1399
Male            1260
Not recorded     441
Name: count, dtype: int64

age_group:
 age_group
Not recorded    1276
45-64            612
65-74            384
75+              377
18-44            333
0-17             118
Name: count, dtype: int64

report_outcome:
 report_outcome
Not serious           1192
Other serious          829
Hospitalization        537
Death                  468
Life-threatening        53
Disability              17
Congenital anomaly       4
Name: count, dtype: int64

Saved reports_clean.csv with shape (3100, 27)


In [5]:
PLACEHOLDERS = {'UNKNOWN', 'UNSPECIFIED', 'NOT SPECIFIED', 'UNK', 'NONE', 'N/A', 'NA'}

def clean_text(s):
    """Uppercase, trim, collapse repeated spaces. Missing stays missing."""
    return (s.astype('string').str.upper().str.strip()
             .str.replace(r'\s+', ' ', regex=True))

# ---------- DRUGS ----------
# Standardized name = first openFDA generic name, else active substance, else reported product name
generic_first = drugs['generic_name'].str.split('|').str[0]
name_raw = generic_first.fillna(drugs['activesubstancename']).fillna(drugs['medicinalproduct'])
drugs['name_source'] = np.select(
    [generic_first.notna(), drugs['activesubstancename'].notna(), drugs['medicinalproduct'].notna()],
    ['openfda_generic_name', 'active_substance', 'reported_product_name'],
    default='missing')
drugs['drug_name_std'] = clean_text(name_raw)

is_placeholder = drugs['drug_name_std'].isin(PLACEHOLDERS)
print('Placeholder drug names set to missing:', is_placeholder.sum())
drugs.loc[is_placeholder, 'drug_name_std'] = pd.NA
print('Drug rows with no usable name:', drugs['drug_name_std'].isna().sum())
print('\nWhere the standardized name came from:\n', drugs['name_source'].value_counts())

ROLE_MAP = {'1': 'Suspect', '2': 'Concomitant', '3': 'Interacting'}
drugs['drug_role'] = drugs['drugcharacterization'].map(ROLE_MAP).fillna('Not recorded')
print('\nDrug roles:\n', drugs['drug_role'].value_counts())

named = drugs.dropna(subset=['drug_name_std'])
print('\nRepeated (report, drug) pairs:', named.duplicated(['safetyreportid', 'drug_name_std']).sum())

# Rows vs unique reports: this is the double-counting check
summary = (named.groupby('drug_name_std')
           .agg(rows=('safetyreportid', 'size'), reports=('safetyreportid', 'nunique'))
           .assign(rows_per_report=lambda d: (d['rows'] / d['reports']).round(2))
           .sort_values('reports', ascending=False))
print('\nTop 15 drug names by UNIQUE REPORTS (any role):')
print(summary.head(15))

suspect = named[named['drug_role'] == 'Suspect']
print('\nTop 10 drug names by unique reports (suspect role only):')
print(suspect.groupby('drug_name_std')['safetyreportid'].nunique()
             .sort_values(ascending=False).head(10))

# ---------- REACTIONS ----------
reactions['reaction_pt'] = clean_text(reactions['reactionmeddrapt'])
print('\nReaction rows with no term:', reactions['reaction_pt'].isna().sum())

OUTCOME_MAP = {'1': 'Recovered/resolved', '2': 'Recovering/resolving',
               '3': 'Not recovered/not resolved', '4': 'Recovered with sequelae',
               '5': 'Fatal', '6': 'Unknown'}
print('Unexpected reaction-outcome codes:',
      set(reactions['reactionoutcome'].dropna()) - set(OUTCOME_MAP))
reactions['reaction_outcome'] = reactions['reactionoutcome'].map(OUTCOME_MAP).fillna('Not recorded')

rx_named = reactions.dropna(subset=['reaction_pt'])
print('Repeated (report, reaction) pairs:', rx_named.duplicated(['safetyreportid', 'reaction_pt']).sum())

rx_summary = (rx_named.groupby('reaction_pt')
              .agg(rows=('safetyreportid', 'size'), reports=('safetyreportid', 'nunique'))
              .sort_values('reports', ascending=False))
print('\nTop 10 reaction terms by unique reports:')
print(rx_summary.head(10))

# ---------- SAVE ----------
drugs.to_csv(DATA / 'processed/drugs_clean.csv', index=False)
reactions.to_csv(DATA / 'processed/reactions_clean.csv', index=False)
print('\nSaved drugs_clean.csv', drugs.shape, '| reactions_clean.csv', reactions.shape)

Placeholder drug names set to missing: 0
Drug rows with no usable name: 0

Where the standardized name came from:
 name_source
openfda_generic_name     9582
active_substance         2113
reported_product_name     161
Name: count, dtype: int64

Drug roles:
 drug_role
Suspect        7719
Concomitant    4094
Interacting      43
Name: count, dtype: int64

Repeated (report, drug) pairs: 3251

Top 15 drug names by UNIQUE REPORTS (any role):
                    rows  reports  rows_per_report
drug_name_std                                     
LEUPROLIDE ACETATE   476      168             2.83
DUPILUMAB            188      139             1.35
ADALIMUMAB           207      134             1.54
EVOLOCUMAB           164      114             1.44
RISANKIZUMAB-RZAA    164      110             1.49
CLOZAPINE             99       96             1.03
ACETAMINOPHEN        116       92             1.26
UPADACITINIB         121       90             1.34
FENTANYL              89       83             1.07


In [6]:
import json

with open(DATA / 'raw/drug_event_sample_2024-01.json') as f:
    raw = json.load(f)

extra = pd.DataFrame([{
    'safetyreportid': r.get('safetyreportid'),
    'safetyreportversion': r.get('safetyreportversion'),
    'n_reportduplicate': len(r.get('reportduplicate') or []),
} for r in raw['records']])
del raw  # free memory

reports = reports.merge(extra, on='safetyreportid', how='left', validate='one_to_one')

print('Report versions:\n', reports['safetyreportversion'].value_counts(dropna=False))
print('\nduplicate flag:\n', reports['duplicate_flag'].value_counts(dropna=False))
print('\nReports listing duplicate identifiers (n_reportduplicate > 0):',
      (reports['n_reportduplicate'] > 0).sum())

same_company = reports[reports['companynumb'].notna()
                       & reports['companynumb'].duplicated(keep=False)]
print('\nMissing companynumb:', reports['companynumb'].isna().sum())
print('Reports sharing a companynumb with another report in the sample:', len(same_company))
print(same_company.sort_values('companynumb')
      [['safetyreportid', 'companynumb', 'receivedate', 'safetyreportversion']].head(10))

flag_cols = ['seriousnessdeath', 'seriousnesslifethreatening', 'seriousnesshospitalization',
             'seriousnessdisabling', 'seriousnesscongenitalanomali', 'seriousnessother']
print('\nReports with 2+ seriousness flags:',
      (reports[flag_cols].eq('1').sum(axis=1) >= 2).sum())

reports.to_csv(DATA / 'processed/reports_clean.csv', index=False)
print('\nSaved reports_clean.csv', reports.shape)

Report versions:
 safetyreportversion
1     2203
2      538
3      205
4       69
5       40
6       20
7        8
10       5
8        5
11       2
9        2
13       2
12       1
Name: count, dtype: int64

duplicate flag:
 duplicate_flag
NaN    2182
1       918
Name: count, dtype: int64

Reports listing duplicate identifiers (n_reportduplicate > 0): 915

Missing companynumb: 203
Reports sharing a companynumb with another report in the sample: 6
     safetyreportid                       companynumb receivedate  \
172        23355140           FR-AFSSAPS-RN2023001456    20240102   
548        23374775           FR-AFSSAPS-RN2023001456    20240106   
325        23363834  US-MLMSERVICE-20231222-4739054-1    20240104   
1403       23400290  US-MLMSERVICE-20231222-4739054-1    20240115   
400        23367909  US-MLMSERVICE-20231226-4741713-1    20240105   
501        23374728  US-MLMSERVICE-20231226-4741713-1    20240106   

     safetyreportversion  
172                    3  
548        

In [7]:
# ---------- One row per (report, drug) ----------
named_drugs = drugs.dropna(subset=['drug_name_std'])
report_drugs = (named_drugs
    .groupby(['safetyreportid', 'drug_name_std'], as_index=False)
    .agg(n_entries=('drug_seq', 'size'),
         is_suspect=('drug_role', lambda s: (s == 'Suspect').any()),
         is_concomitant=('drug_role', lambda s: (s == 'Concomitant').any()),
         is_interacting=('drug_role', lambda s: (s == 'Interacting').any())))

# ---------- One row per (report, reaction) ----------
named_rx = reactions.dropna(subset=['reaction_pt'])
report_reactions = (named_rx
    .groupby(['safetyreportid', 'reaction_pt'], as_index=False)
    .agg(n_entries=('reaction_seq', 'size'),
         any_fatal=('reaction_outcome', lambda s: (s == 'Fatal').any())))

# ---------- Checks ----------
assert not report_drugs.duplicated(['safetyreportid', 'drug_name_std']).any()
assert not report_reactions.duplicated(['safetyreportid', 'reaction_pt']).any()
assert report_drugs['n_entries'].sum() == len(named_drugs)
assert report_reactions['n_entries'].sum() == len(named_rx)
assert set(report_drugs['safetyreportid']) <= set(reports['safetyreportid'])
print('All checks passed.')

print('Drug rows:', len(named_drugs), '->', len(report_drugs), 'unique (report, drug) rows')
print('Reaction rows:', len(named_rx), '->', len(report_reactions), 'unique (report, reaction) rows')
print('Drugs listed as BOTH suspect and concomitant in one report:',
      (report_drugs['is_suspect'] & report_drugs['is_concomitant']).sum())
print('Reports with at least one suspect drug:',
      report_drugs.loc[report_drugs['is_suspect'], 'safetyreportid'].nunique())

report_drugs.to_csv(DATA / 'processed/report_drugs.csv', index=False)
report_reactions.to_csv(DATA / 'processed/report_reactions.csv', index=False)
print('Saved report_drugs.csv and report_reactions.csv')

All checks passed.
Drug rows: 11856 -> 8605 unique (report, drug) rows
Reaction rows: 9335 -> 9217 unique (report, reaction) rows
Drugs listed as BOTH suspect and concomitant in one report: 31
Reports with at least one suspect drug: 3086
Saved report_drugs.csv and report_reactions.csv


In [8]:
import json

def as_list(x):
    """openFDA sometimes returns one object where other records return a list."""
    if x is None:
        return []
    return x if isinstance(x, list) else [x]

with open(DATA / 'raw/drug_event_sample_2024-01.json') as f:
    raw = json.load(f)

# ---------- 0. Confirm the cause of the error ----------
print('Type of the reportduplicate field across reports:')
print(pd.Series([type(r.get('reportduplicate')).__name__ for r in raw['records']]).value_counts())

dup = pd.DataFrame([
    {'safetyreportid': r.get('safetyreportid'),
     'duplicatesource': d.get('duplicatesource'),
     'duplicatenumb': d.get('duplicatenumb')}
    for r in raw['records']
    for d in as_list(r.get('reportduplicate'))
    if isinstance(d, dict)
])
del raw

# Fix the earlier n_reportduplicate column (entries per report, counted correctly)
entry_counts = dup.groupby('safetyreportid').size()
reports['n_reportduplicate'] = reports['safetyreportid'].map(entry_counts).fillna(0).astype(int)
print('\nReports with duplicate identifiers (corrected):', (reports['n_reportduplicate'] > 0).sum())
reports.to_csv(DATA / 'processed/reports_clean.csv', index=False)

# ---------- 1. Do 'duplicate' identifiers point at other records in OUR sample? ----------
own_company = dup['safetyreportid'].map(reports.set_index('safetyreportid')['companynumb'])
dup['is_own_company_number'] = dup['duplicatenumb'] == own_company
sample_keys = set(reports['safetyreportid']) | set(reports['companynumb'].dropna())
dup['points_to_other_sample_report'] = (dup['duplicatenumb'].isin(sample_keys)
                                        & ~dup['is_own_company_number']
                                        & (dup['duplicatenumb'] != dup['safetyreportid']))

print('\nDuplicate-identifier entries:', len(dup), 'across', dup['safetyreportid'].nunique(), 'reports')
print('\nBy source (top 10):\n', dup['duplicatesource'].value_counts().head(10))
print("\nEntries equal to the report's own company number:", dup['is_own_company_number'].sum())
print('Entries pointing to a DIFFERENT report in this sample:', dup['points_to_other_sample_report'].sum())

# ---------- 2. What do follow-up versions mean for the dates? ----------
is_followup = reports['safetyreportversion'].astype(int) > 1
receipt_later = reports['receiptdate_dt'] > reports['receivedate_dt']
print('\nVersion > 1 vs. receiptdate later than receivedate:')
print(pd.crosstab(is_followup.rename('version > 1'),
                  receipt_later.rename('receiptdate after receivedate')))

# ---------- 3. Are the top drugs concentrated on a few days? ----------
top5 = summary.head(5).index
chk = (report_drugs[report_drugs['drug_name_std'].isin(top5)]
       .merge(reports[['safetyreportid', 'receivedate_dt']], on='safetyreportid'))
per_day = chk.groupby(['drug_name_std', 'receivedate_dt']).size()
print('\nTop-5 drugs: unique reports, days with a report, most reports on one day:')
print(per_day.groupby('drug_name_std')
      .agg(reports='sum', days='count', busiest_day='max')
      .sort_values('reports', ascending=False))

Type of the reportduplicate field across reports:
NoneType    2185
dict         543
list         372
Name: count, dtype: int64

Reports with duplicate identifiers (corrected): 915

Duplicate-identifier entries: 1945 across 915 reports

By source (top 10):
 duplicatesource
SA                                        211
PORTAL ID                                 200
AMGEN INC                                 196
MHRA                                       81
PFIZER INC                                 79
MC USA                                     55
EMA                                        45
Market Authorization Holder AER Number     39
Adverse Reaction Report Number             37
SANOFI                                     30
Name: count, dtype: int64

Entries equal to the report's own company number: 270
Entries pointing to a DIFFERENT report in this sample: 3

Version > 1 vs. receiptdate later than receivedate:
receiptdate after receivedate  False  True 
version > 1                     

## Cleaning decisions
- Counting unit: one report = one `safetyreportid`. Not patients.
- Tables: reports (1 row/report), report_drugs (1 row/report-drug), report_reactions (1 row/report-reaction).
- Repeated (report, drug) pairs and (report, reaction) pairs collapsed to one row, so counts use unique reports.
- Drug name: first openFDA generic name, else active substance, else reported product name (uppercased).
- Drug role: suspect is used for main drug analyses; any-role is shown only when labelled.
- Age converted to years using the unit code; ages outside 0-120 set to missing.
- Report outcome: one category per report, highest of death > life-threatening > hospitalization > disability > congenital anomaly > other serious.
- Reaction outcome (per reaction) is kept separate from report outcome.
- Duplicate-flagged and follow-up reports are kept; sensitivity checks planned.